# Step 4a: Basic Claim-Level Feature Engineering

Builds simple, directly-interpretable features **at the claim grain** (one
row per claim) — as opposed to `04b_feature_engineering.ipynb`, which
aggregates all the way up to one row per provider. These claim-level
features are grounded in `03a_claim_level_eda.ipynb`'s findings.

All 25 features are computed by `build_claim_level_features()` in
`src/features.py` — the single source of truth also used internally by
`build_provider_features()`, so this notebook and the provider-level
pipeline can never drift out of sync. This notebook's job is exploring and
documenting what that function builds, then saving the result for
`03b_provider_level_eda.ipynb`'s broader provider-level aggregation to
consume.

In [1]:
import sys
sys.path.append('..')

import pandas as pd

from src.data_loader import build_claim_level_dataset
from src.cleaning import clean_claim_level_dataset
from src.features import build_claim_level_features

df = build_claim_level_features(clean_claim_level_dataset(build_claim_level_dataset()))
df.shape

(558211, 86)

## 1. Claim Type

In [2]:
df['IsInpatient'].value_counts()

IsInpatient
0    517737
1     40474
Name: count, dtype: int64

*`IsInpatient`: 1 if the claim is Inpatient, else 0 — a numeric version of `ClaimType`, ready for models that need 0/1 rather than a string.*

## 2. Financial

In [3]:
df[['LogReimbursement', 'HasDeductible']].describe()

,LogReimbursement,HasDeductible
count,558211.000000,558211.000000
mean,4.752708,0.110191
std,1.952547,0.313128
min,0.000000,0.000000
25%,3.713572,0.000000
50%,4.394449,0.000000
75%,5.707110,0.000000
max,11.736077,1.000000


*`LogReimbursement`: log(1 + reimbursed amount) — tames the heavy right skew found in Step 3a.
`HasDeductible`: 1 if any deductible was paid, else 0 — turns the bimodal (mostly-0-or-~1068) deductible pattern into a clean binary flag.*

## 3. Duration

In [4]:
df[['ClaimDuration', 'SameDayClaim', 'LengthOfStay']].describe()

,ClaimDuration,SameDayClaim,LengthOfStay
count,558211.000000,558211.000000,40474.000000
mean,1.727940,0.813228,5.665168
std,4.904984,0.389728,5.638538
min,0.000000,0.000000,0.000000
25%,0.000000,1.000000,2.000000
50%,0.000000,1.000000,4.000000
75%,0.000000,1.000000,7.000000
max,36.000000,1.000000,35.000000


*`ClaimDuration`: days from claim start to end (all claims).
`SameDayClaim`: 1 if duration is 0 — 81.3% of claims per Step 3a.
`LengthOfStay`: days admitted to discharged — Inpatient-only, NaN for Outpatient (structural, not missing data).*

## 4. Diagnosis

In [5]:
df[['NumDiagnosisCodes', 'HasDiagnosis']].describe()

,NumDiagnosisCodes,HasDiagnosis
count,558211.000000,558211.000000
mean,3.010897,0.981274
std,2.448213,0.135555
min,0.000000,0.000000
25%,1.000000,1.000000
50%,2.000000,1.000000
75%,4.000000,1.000000
max,10.000000,1.000000


*`NumDiagnosisCodes`: count of non-null diagnosis code slots used (0-10) — coding intensity.
`HasDiagnosis`: 1 if any diagnosis code is present (99.98% of claims per Step 3a).
`PrimaryDiagnosis`: friendly alias for `ClmDiagnosisCode_1`, the main diagnosis code.*

## 5. Procedure

In [6]:
df[['NumProcedureCodes', 'HasProcedure']].describe()

,NumProcedureCodes,HasProcedure
count,558211.000000,558211.000000
mean,0.053557,0.041758
std,0.280534,0.200037
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,5.000000,1.000000


*`NumProcedureCodes`: count of non-null procedure code slots used (0-6).
`HasProcedure`: 1 if any procedure code is present — only 4.2% of claims overall, but 57.2% of Inpatient claims (Step 3a) — this flag carries most of the information `NumProcedureCodes` does, more robustly.*

## 6. Claim Complexity

In [7]:
df['ClaimComplexity'].describe()

count    558211.000000
mean          3.064454
std           2.574540
min           0.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          15.000000
Name: ClaimComplexity, dtype: float64

*`ClaimComplexity`: total diagnosis + procedure codes on the claim — a single, simple measure of how much coding activity a claim involved.*

## 7. Physicians

In [8]:
# NumPhysiciansInvolved already exists — derived in src/cleaning.py's
# add_physician_involvement_count(). HasOperatingPhysician, HasOtherPhysician,
# and SameAttendingOperating are built by build_claim_level_features() above.
df[['NumPhysiciansInvolved', 'HasOperatingPhysician', 'HasOtherPhysician', 'SameAttendingOperating']].mean().round(3)

NumPhysiciansInvolved     1.560
HasOperatingPhysician     0.205
HasOtherPhysician         0.358
SameAttendingOperating    0.110
dtype: float64

*`NumPhysiciansInvolved` (already built in `src/cleaning.py`): count of Attending/Operating/Other physicians present (0-3).
`HasOperatingPhysician` / `HasOtherPhysician`: binary presence flags for the two structurally-sparse physician roles.
`SameAttendingOperating`: 1 if the same physician is listed as both attending and operating (10.96% of claims per Step 3a) — a possible self-referral signal.*

## 8. Beneficiary Health

In [9]:
# AgeAtClaim already exists (src/cleaning.py's add_age_columns()).
# NumChronicConditions and HasRenalDisease are built by
# build_claim_level_features() above.
df[['AgeAtClaim', 'NumChronicConditions', 'HasRenalDisease']].describe()

,AgeAtClaim,NumChronicConditions,HasRenalDisease
count,558211.000000,558211.000000,558211.000000
mean,73.297705,4.498616,0.196786
std,13.017286,2.332301,0.397569
min,25.092402,0.000000,0.000000
25%,67.542779,3.000000,0.000000
50%,74.203970,5.000000,0.000000
75%,81.954825,6.000000,0.000000
max,100.993840,11.000000,1.000000


*`AgeAtClaim` (already built): beneficiary's age in years at the time of this claim.
`NumChronicConditions`: sum of the 11 `ChronicCond_*` flags (0-11) — overall comorbidity burden.
`HasRenalDisease`: alias of the already-cleaned `RenalDiseaseIndicator` (1/0).*

## 9. Temporal

In [10]:
df[['ClaimMonth', 'ClaimDayOfWeek', 'IsWeekend']].describe()

,ClaimMonth,ClaimDayOfWeek,IsWeekend
count,558211.000000,558211.000000,558211.000000
mean,6.354835,2.998508,0.285399
std,3.411155,1.999092,0.451605
min,1.000000,0.000000,0.000000
25%,3.000000,1.000000,0.000000
50%,6.000000,3.000000,0.000000
75%,9.000000,5.000000,1.000000
max,12.000000,6.000000,1.000000


*`ClaimMonth`: calendar month (1-12) of the claim start date.
`ClaimDayOfWeek`: numeric day of week (0=Monday, 6=Sunday), model-friendly.
`IsWeekend`: 1 if Saturday/Sunday — Step 3a found this is close to the 28.6% uniform-week baseline, so expect limited standalone signal.*

## 10. Repeat Claiming

In [11]:
df[['DaysSincePreviousClaim', 'ClaimWithin7Days', 'ClaimWithin30Days']].describe()

,DaysSincePreviousClaim,ClaimWithin7Days,ClaimWithin30Days
count,419655.000000,558211.000000,558211.000000
mean,50.583441,0.112601,0.353769
std,52.177937,0.316104,0.478139
min,0.000000,0.000000,0.000000
25%,13.000000,0.000000,0.000000
50%,33.000000,0.000000,0.000000
75%,70.000000,0.000000,1.000000
max,369.000000,1.000000,1.000000


*`DaysSincePreviousClaim`: gap in days since this beneficiary's prior claim — NaN for a beneficiary's first claim (structural, not missing data).
`ClaimWithin7Days` / `ClaimWithin30Days`: 1 if that gap is <=7 / <=30 days (NaN treated as 0 — no prior claim means no rapid repeat) — Step 3a found 11.3% / 35.4% of claims qualify.*

## 11. Same-Day Duplication

In [12]:
df[['SameBeneProviderDay', 'NumClaimsSameBeneProviderDay']].describe()

,SameBeneProviderDay,NumClaimsSameBeneProviderDay
count,558211.000000,558211.000000
mean,0.005012,1.005066
std,0.070621,0.071750
min,0.000000,1.000000
25%,0.000000,1.000000
50%,0.000000,1.000000
75%,0.000000,1.000000
max,1.000000,3.000000


*`SameBeneProviderDay`: 1 if this claim shares BeneID + Provider + ClaimStartDt with at least one other claim (2,798 such claims per Step 3a).
`NumClaimsSameBeneProviderDay`: exact count of claims sharing that same combination — magnitude behind the flag above.*

## Save

In [13]:
df = df.sort_values('ClaimID').reset_index(drop=True)
df.to_csv('../data/processed/claims_with_basic_features.csv', index=False)
df.shape

(558211, 86)

*Saved to `data/processed/claims_with_basic_features.csv` — the cleaned claim-level table plus all features built above, one row per claim.*

## Summary: What Each Feature Does

| Feature | What it does |
|---|---|
| `IsInpatient` | 1 if the claim is Inpatient, else 0 |
| `LogReimbursement` | log(1 + `InscClaimAmtReimbursed`) — tames right skew |
| `HasDeductible` | 1 if any deductible was paid |
| `ClaimDuration` | Days from `ClaimStartDt` to `ClaimEndDt` |
| `SameDayClaim` | 1 if `ClaimDuration` is 0 |
| `LengthOfStay` | Days admitted to discharged (Inpatient-only, NaN otherwise) |
| `NumDiagnosisCodes` | Count of non-null diagnosis code slots (0-10) |
| `HasDiagnosis` | 1 if any diagnosis code is present |
| `PrimaryDiagnosis` | Alias of `ClmDiagnosisCode_1` |
| `NumProcedureCodes` | Count of non-null procedure code slots (0-6) |
| `HasProcedure` | 1 if any procedure code is present |
| `ClaimComplexity` | `NumDiagnosisCodes` + `NumProcedureCodes` |
| `NumPhysiciansInvolved` | Count of Attending/Operating/Other physicians present (already in `src/cleaning.py`) |
| `HasOperatingPhysician` | 1 if an operating physician is listed |
| `HasOtherPhysician` | 1 if an "other" physician is listed |
| `SameAttendingOperating` | 1 if the same physician is both attending and operating |
| `AgeAtClaim` | Beneficiary's age at the claim date (already in `src/cleaning.py`) |
| `NumChronicConditions` | Sum of the 11 `ChronicCond_*` flags (0-11) |
| `HasRenalDisease` | Alias of the cleaned `RenalDiseaseIndicator` |
| `ClaimMonth` | Calendar month (1-12) of the claim |
| `ClaimDayOfWeek` | Day of week, 0=Monday to 6=Sunday |
| `IsWeekend` | 1 if Saturday/Sunday |
| `DaysSincePreviousClaim` | Days since this beneficiary's previous claim (NaN if first claim) |
| `ClaimWithin7Days` | 1 if the previous claim was within 7 days |
| `ClaimWithin30Days` | 1 if the previous claim was within 30 days |
| `SameBeneProviderDay` | 1 if this claim shares beneficiary + provider + date with another claim |
| `NumClaimsSameBeneProviderDay` | Count of claims sharing that same beneficiary + provider + date |